In [ ]:
# The button "Open NICOS" of the catalogue (?instrument=odin or loki). In the container: asks the control room
# (container/control_room.sh, on the remote screen) for NICOS of this instrument, then shows the remote screen.
# In the local app (container/run_app_local.ps1, TWIN_APP_LOCAL): opens NICOS as windows of this computer
# (container/control_room_local.py).
import os, subprocess, sys
from urllib.parse import parse_qs
from IPython.display import HTML, display
instr = parse_qs(os.environ.get("QUERY_STRING", "")).get("instrument", [""])[0].lower()
instr = instr if instr in ("odin", "loki") else ""
name = {"odin": "ODIN", "loki": "LoKI"}.get(instr, "the instrument")
if os.environ.get("TWIN_APP_LOCAL"):
    flags = 0
    if os.name == "nt":   # no console window; independent of this page (closing the page does not stop NICOS)
        flags = subprocess.CREATE_NO_WINDOW | subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.CREATE_BREAKAWAY_FROM_JOB
    cmd = [sys.executable, os.path.join(os.path.abspath(".."), "container", "control_room_local.py")]
    cmd += ["--instrument", instr] if instr else []
    try:
        subprocess.Popen(cmd, cwd=os.path.abspath(".."), creationflags=flags,
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except OSError:      # (breakaway not allowed: without it)
        subprocess.Popen(cmd, cwd=os.path.abspath(".."), creationflags=flags & ~0x01000000,
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    display(HTML(f"""<div style="font-family: sans-serif; max-width: 640px; margin: 40px">
<h2>NICOS of {name} opens on your desktop</h2>
<p>In about a minute the NICOS window is there, logged in. Close it to stop NICOS. You can close this page now.</p>
<p style="color:#6b7280">If NICOS is already open, nothing new opens: use the window you have.</p></div>"""))
else:
    if instr:
        with open(os.path.expanduser("~/.control_room_request"), "w") as f:
            f.write(instr)
    display(HTML(f"""<div style="font-family: sans-serif; max-width: 640px; margin: 40px">
<h2>The control room of {name}</h2><p>Opening... NICOS is there, logged in, in about a minute.</p></div>
<script>window.location.replace("../../../desktop/");</script>"""))